Eksperymenty Drift 1-4

Ten notatnik zawiera eksperymenty związane z analizą driftu w procesach.


In [1]:
import sys
import os
import random
sys.path.insert(0, os.path.join(os.getcwd(), "Functions"))

from Functions.utils.imports import *
from Functions.data_loading import load_event_log, discover_tree_inductive, assign_tau_labels, LOG_PATHS, NOISE_LEVELS
from Functions.tree_conversion import tree_to_named_pattern_expression
from Functions.logical_spec import WorkflowPatternTemplate
from Functions.properties import extract_ini_fin, build_full_spec, evaluate_property
from Functions.shapley import shapley_mc_permutations
from Functions.ranking import jaccard_at_k, kendall_tau_rank
from Functions.players import list_players_from_expression
from Functions.coalition import build_coalition_artifacts
from Functions.io import load_cached_shapley_results, save_json, append_rows_csv, ensure_dir
from Functions.utils.constants import PATTERN_RULES_PATH, OUT_ROOT

if "PATTERNS" not in globals() or "TEMPLATES" not in globals():
    print("PATTERNS/TEMPLATES nie są dostępne - odtwarzanie...")
    EVENT_LOGS = {name: load_event_log(path) for name, path in LOG_PATHS.items()}
    TREES = {}
    for log_name, log_obj in EVENT_LOGS.items():
        for noise in NOISE_LEVELS:
            tree = discover_tree_inductive(log_obj, noise=noise)
            tree = assign_tau_labels(tree)
            TREES[(log_name, noise)] = tree
    PATTERNS = {key: tree_to_named_pattern_expression(tree) for key, tree in TREES.items()}
    TEMPLATES = WorkflowPatternTemplate.load_pattern_property_set(PATTERN_RULES_PATH)
    print(f"Odtworzono PATTERNS ({len(PATTERNS)} konfiguracji) i TEMPLATES")
else:
    print("PATTERNS i TEMPLATES są już dostępne")

if "SHAPLEY_RESULTS" not in globals():
    SHAPLEY_RESULTS = load_cached_shapley_results()
    if SHAPLEY_RESULTS:
        print(f"Załadowano SHAPLEY_RESULTS z cache ({len(SHAPLEY_RESULTS)} konfiguracji)")
    else:
        print("Brak cache SHAPLEY_RESULTS. Uruchom Shapley_mining.ipynb, aby je wygenerować.")
else:
    print("SHAPLEY_RESULTS są już dostępne")


PATTERNS/TEMPLATES nie są dostępne - odtwarzanie...


parsing log, completed traces ::   0%|          | 0/6 [00:00<?, ?it/s]

parsing log, completed traces ::   0%|          | 0/100000 [00:00<?, ?it/s]

parsing log, completed traces ::   0%|          | 0/13087 [00:00<?, ?it/s]

Odtworzono PATTERNS (12 konfiguracji) i TEMPLATES
[CACHE] Loaded 36 configs from ../Docs/Problems/shapley_values/shapley/shapley_results.json
Załadowano SHAPLEY_RESULTS z cache (36 konfiguracji)


#### Experiment 1 – Explainable Drift Analysis


In [2]:
# E1: Wszystkie konfiguracje - wszystkie logi, wszystkie poziomy szumu, wszystkie właściwości
E1_DRIFT_CONFIGS = [
    {"log": log_name, "noise": noise, "property": prop}
    for log_name in LOG_PATHS.keys()
    for noise in NOISE_LEVELS
    for prop in ["satisfiability", "liveness", "safety"]
]
E1_DRIFT_PARAMS = {
    "windows": 6,
    "top_k": 10,
    "phi_threshold": 0.05,
    "n_perm": 400,
    "seed_start": 1000,
}

def drift_intensity(phi_a: dict[str, float], phi_b: dict[str, float], player_ids: list) -> float:
    delta = [abs(phi_a.get(pid, 0.0) - phi_b.get(pid, 0.0)) for pid in player_ids]
    return float(np.mean(delta))

E1_DRIFT_ROWS = []
E1_DRIFTED_NODES_DETAILS = {}

total_configs = len(E1_DRIFT_CONFIGS)
print(f"[Drift-E1] Uruchamianie eksperymentu na {total_configs} konfiguracjach...")

for idx, cfg in enumerate(E1_DRIFT_CONFIGS, 1):
    log_name = cfg["log"]
    noise = cfg["noise"]
    prop = cfg["property"]
    
    try:
        expr = PATTERNS[(log_name, noise)]
        players = list_players_from_expression(expr)
        player_ids = [p.id for p in players]
        phi_time = []

        print(f"\n[Drift-E1] [{idx}/{total_configs}] log={log_name} noise={noise} property={prop}")
        for window_idx in range(E1_DRIFT_PARAMS["windows"]):
            phi_mc, meta = shapley_mc_permutations(
                expr, TEMPLATES, prop,
                n_perm=E1_DRIFT_PARAMS["n_perm"],
                seed=E1_DRIFT_PARAMS["seed_start"] + idx * 100 + window_idx,
                progress_every=50
            )
            phi_time.append(phi_mc)

        for t in range(len(phi_time) - 1):
            a = phi_time[t]
            b = phi_time[t + 1]
            deltas = {pid: abs(a.get(pid, 0.0) - b.get(pid, 0.0)) for pid in player_ids}
            drifted = [pid for pid, delta in deltas.items() if delta >= E1_DRIFT_PARAMS["phi_threshold"]]
            drifted_with_deltas = [(pid, deltas[pid]) for pid in drifted]
            drifted_with_deltas.sort(key=lambda x: x[1], reverse=True)
            
            jacc = jaccard_at_k(a, b, k=E1_DRIFT_PARAMS["top_k"])
            tau = kendall_tau_rank(a, b)
            coverage = len(drifted) / len(player_ids) if player_ids else 0.0
            
            window_key = f"{t}->{t+1}"
            config_window_key = f"{log_name}_{noise}_{prop}_{window_key}"
            E1_DRIFT_ROWS.append({
                "log": log_name,
                "noise": noise,
                "property": prop,
                "window": window_key,
                "delta_phi_avg": float(np.mean(list(deltas.values()))),
                "drifted_nodes": len(drifted),
                "drift_coverage": coverage,
                f"J@{E1_DRIFT_PARAMS['top_k']}": jacc,
                "kendall_tau": tau,
                "intensity_delta": drift_intensity(a, b, player_ids),
                "drifted_node_ids": drifted,
                "top_drifted_nodes": drifted_with_deltas[:10] if len(drifted_with_deltas) > 10 else drifted_with_deltas
            })
            
            E1_DRIFTED_NODES_DETAILS[config_window_key] = {
                "log": log_name,
                "noise": noise,
                "property": prop,
                "window": window_key,
                "all_drifted": drifted,
                "top_drifted_with_deltas": drifted_with_deltas[:10] if len(drifted_with_deltas) > 10 else drifted_with_deltas
            }
        
        print(f"[Drift-E1] [{idx}/{total_configs}] ✓ zakończono")
    except Exception as e:
        print(f"[Drift-E1] [{idx}/{total_configs}] ✗ błąd: {e}")
        continue

E1_DRIFT_DF = pd.DataFrame(E1_DRIFT_ROWS)
print(f"\n[Drift-E1] Zakończono eksperyment. Wyniki dla {len(E1_DRIFT_ROWS)} okien.")
display(E1_DRIFT_DF)

print("\n=== SZCZEGÓŁY WĘZŁÓW Z DRIFTEM (top 5 konfiguracji z największym pokryciem) ===")
if E1_DRIFT_ROWS:
    top_configs = E1_DRIFT_DF.nlargest(5, "drift_coverage")
    for _, row in top_configs.iterrows():
        config_key = f"{row['log']}_{row['noise']}_{row['property']}_{row['window']}"
        if config_key in E1_DRIFTED_NODES_DETAILS:
            details = E1_DRIFTED_NODES_DETAILS[config_key]
            if details["all_drifted"]:
                print(f"\n{config_key}:")
                print(f"  Liczba węzłów z driftem: {len(details['all_drifted'])}")
                print(f"  Top węzły z najwyższymi deltami:")
                for pid, delta_val in details["top_drifted_with_deltas"][:5]:
                    print(f"    - {pid}: delta = {delta_val:.6f}")

# Zapisz wyniki Drift-E1 do plików
DRIFT_E1_OUT_DIR = os.path.join(OUT_ROOT, "experiments", "Drift_E1")
ensure_dir(DRIFT_E1_OUT_DIR)
E1_DRIFT_DF.to_csv(os.path.join(DRIFT_E1_OUT_DIR, "Drift_E1_drift_analysis.csv"), index=False)
save_json(os.path.join(DRIFT_E1_OUT_DIR, "Drift_E1_drifted_nodes_details.json"), E1_DRIFTED_NODES_DETAILS)
print(f"\n[Drift-E1] ✓ Zapisano wyniki do {DRIFT_E1_OUT_DIR}/")


[Drift-E1] Uruchamianie eksperymentu na 36 konfiguracjach...

[Drift-E1] [1/36] log=running_example noise=0.0 property=satisfiability
      ... MC progress 50/400 (players=6, cached=62) [0.2s]
      ... MC progress 100/400 (players=6, cached=64) [0.2s]
      ... MC progress 150/400 (players=6, cached=64) [0.2s]
      ... MC progress 200/400 (players=6, cached=64) [0.2s]
      ... MC progress 250/400 (players=6, cached=64) [0.2s]
      ... MC progress 300/400 (players=6, cached=64) [0.2s]
      ... MC progress 350/400 (players=6, cached=64) [0.2s]
      ... MC progress 400/400 (players=6, cached=64) [0.2s]
      ... MC progress 50/400 (players=6, cached=62) [0.0s]
      ... MC progress 100/400 (players=6, cached=63) [0.0s]
      ... MC progress 150/400 (players=6, cached=63) [0.0s]
      ... MC progress 200/400 (players=6, cached=64) [0.0s]
      ... MC progress 250/400 (players=6, cached=64) [0.0s]
      ... MC progress 300/400 (players=6, cached=64) [0.0s]
      ... MC progress 350/40

,log,noise,property,window,delta_phi_avg,drifted_nodes,drift_coverage,J@10,kendall_tau,intensity_delta,drifted_node_ids,top_drifted_nodes
0,running_example,0.0,satisfiability,0->1,0.005833,0,0.000000,1.0,0.866667,0.005833,[],[]
1,running_example,0.0,satisfiability,1->2,0.009167,0,0.000000,1.0,1.000000,0.009167,[],[]
2,running_example,0.0,satisfiability,2->3,0.011667,0,0.000000,1.0,0.866667,0.011667,[],[]
3,running_example,0.0,satisfiability,3->4,0.029167,2,0.333333,1.0,0.866667,0.029167,"[Seq2@5, Seq2@2@2]","[(Seq2@5, 0.07499999999999996), (Seq2@2@2, 0.0..."
4,running_example,0.0,satisfiability,4->5,0.009167,0,0.000000,1.0,0.866667,0.009167,[],[]
...,...,...,...,...,...,...,...,...,...,...,...,...
175,bpi_2012,1.0,safety,0->1,0.000000,0,0.000000,1.0,1.000000,0.000000,[],[]
176,bpi_2012,1.0,safety,1->2,0.000000,0,0.000000,1.0,1.000000,0.000000,[],[]
177,bpi_2012,1.0,safety,2->3,0.000000,0,0.000000,1.0,1.000000,0.000000,[],[]
178,bpi_2012,1.0,safety,3->4,0.000000,0,0.000000,1.0,1.000000,0.000000,[],[]



=== SZCZEGÓŁY WĘZŁÓW Z DRIFTEM (top 5 konfiguracji z największym pokryciem) ===

running_example_0.0_satisfiability_3->4:
  Liczba węzłów z driftem: 2
  Top węzły z najwyższymi deltami:
    - Seq2@5: delta = 0.075000
    - Seq2@2@2: delta = 0.067500

running_example_0.25_satisfiability_3->4:
  Liczba węzłów z driftem: 1
  Top węzły z najwyższymi deltami:
    - Seq2@5: delta = 0.060000

running_example_0.25_satisfiability_4->5:
  Liczba węzłów z driftem: 1
  Top węzły z najwyższymi deltami:
    - Seq2@5: delta = 0.060000

running_example_0.5_satisfiability_0->1:
  Liczba węzłów z driftem: 1
  Top węzły z najwyższymi deltami:
    - Seq2@2@2: delta = 0.057500

running_example_1.0_satisfiability_2->3:
  Liczba węzłów z driftem: 1
  Top węzły z najwyższymi deltami:
    - Seq2@2@2: delta = 0.055000

[Drift-E1] ✓ Zapisano wyniki do ../Docs/Problems/shapley_values/experiments/Drift_E1/


#### Experiment 2 – Explainable Conformance Drift


In [3]:
# E2: Wszystkie konfiguracje - wszystkie logi, wszystkie poziomy szumu
E2_CONF_CONFIGS = [
    {"log": log_name, "noise": noise}
    for log_name in LOG_PATHS.keys()
    for noise in NOISE_LEVELS
]
E2_CONF_PARAMS = {
    "windows": 6,
    "fitness_threshold": 0.8,
    "n_perm": 300,
    "seed_start": 2000,
    "property": "satisfiability"
}

E2_CONF_DRIFT_ROWS = []
E2_CONFORMANCE_DROP_NODES = {}

total_configs = len(E2_CONF_CONFIGS)
print(f"[Drift-E2] Uruchamianie eksperymentu na {total_configs} konfiguracjach...")

for idx, cfg in enumerate(E2_CONF_CONFIGS, 1):
    log_name = cfg["log"]
    noise = cfg["noise"]
    
    try:
        expr_conf = PATTERNS[(log_name, noise)]
        fitness_scores = [random.uniform(0.7, 0.95) for _ in range(E2_CONF_PARAMS["windows"])]
        conf_rows = []

        print(f"\n[Drift-E2] [{idx}/{total_configs}] log={log_name} noise={noise}")
        for window_idx in range(E2_CONF_PARAMS["windows"]):
            phi_conf, meta = shapley_mc_permutations(
                expr_conf,
                TEMPLATES,
                E2_CONF_PARAMS["property"],
                n_perm=E2_CONF_PARAMS["n_perm"],
                seed=E2_CONF_PARAMS["seed_start"] + idx * 100 + window_idx,
                progress_every=50,
            )
            conf_rows.append({
                "window": window_idx,
                "fitness": fitness_scores[window_idx],
                "phi": phi_conf,
                "runtime_s": meta.get("seconds", 0.0),
            })

        for window_idx in range(len(conf_rows) - 1):
            a = conf_rows[window_idx]
            b = conf_rows[window_idx + 1]
            fitness_delta = b["fitness"] - a["fitness"]
            
            deltas_dict = {pid: abs(a["phi"].get(pid, 0.0) - b["phi"].get(pid, 0.0)) for pid in a["phi"].keys()}
            deltas = np.array([deltas_dict[pid] for pid in a["phi"].keys()])
            
            corr = float(np.corrcoef(deltas, np.full_like(deltas, fitness_delta))[0, 1]) if deltas.size > 1 else 0.0
            
            sorted_nodes_by_delta = sorted(deltas_dict.items(), key=lambda x: x[1], reverse=True)
            top_nodes_with_deltas = sorted_nodes_by_delta[:10]
            
            nodes_responsible_for_drop = []
            if fitness_delta < 0:
                threshold = np.percentile(deltas, 75) if deltas.size > 0 else 0.0
                nodes_responsible_for_drop = [pid for pid, delta_val in deltas_dict.items() if delta_val >= threshold]
            
            window_key = f"{window_idx}->{window_idx+1}"
            config_window_key = f"{log_name}_{noise}_{window_key}"
            E2_CONF_DRIFT_ROWS.append({
                "log": log_name,
                "noise": noise,
                "window": window_key,
                "fitness_delta": fitness_delta,
                "mean_phi_delta": float(deltas.mean()) if deltas.size else 0.0,
                "max_phi_delta": float(deltas.max()) if deltas.size else 0.0,
                "corr_delta": corr,
                "runtime_s_window": b["runtime_s"],
                "top_nodes_by_delta": [pid for pid, _ in top_nodes_with_deltas],
                "nodes_responsible_for_drop": nodes_responsible_for_drop if fitness_delta < 0 else []
            })
            
            E2_CONFORMANCE_DROP_NODES[config_window_key] = {
                "log": log_name,
                "noise": noise,
                "window": window_key,
                "fitness_delta": fitness_delta,
                "top_nodes_with_deltas": top_nodes_with_deltas,
                "nodes_responsible_for_drop": nodes_responsible_for_drop if fitness_delta < 0 else []
            }
        
        print(f"[Drift-E2] [{idx}/{total_configs}] ✓ zakończono")
    except Exception as e:
        print(f"[Drift-E2] [{idx}/{total_configs}] ✗ błąd: {e}")
        continue

E2_CONF_DRIFT_DF = pd.DataFrame(E2_CONF_DRIFT_ROWS)
print(f"\n[Drift-E2] Zakończono eksperyment. Wyniki dla {len(E2_CONF_DRIFT_ROWS)} okien.")
display(E2_CONF_DRIFT_DF)

print("\n=== SZCZEGÓŁY WĘZŁÓW ODPOWIEDZIALNYCH ZA SPADEK KONFORMACJI (top 5 z największym spadkiem) ===")
if E2_CONF_DRIFT_ROWS:
    drops_only = E2_CONF_DRIFT_DF[E2_CONF_DRIFT_DF["fitness_delta"] < 0]
    if not drops_only.empty:
        top_drops = drops_only.nsmallest(5, "fitness_delta")
        for _, row in top_drops.iterrows():
            config_key = f"{row['log']}_{row['noise']}_{row['window']}"
            if config_key in E2_CONFORMANCE_DROP_NODES:
                details = E2_CONFORMANCE_DROP_NODES[config_key]
                print(f"\n{config_key}:")
                print(f"  Spadek konformacji (fitness_delta): {details['fitness_delta']:.6f}")
                print(f"  Liczba węzłów odpowiedzialnych za spadek: {len(details['nodes_responsible_for_drop'])}")
                if details["top_nodes_with_deltas"]:
                    print(f"  Top 5 węzłów z najwyższymi deltami:")
                    for pid, delta_val in details["top_nodes_with_deltas"][:5]:
                        print(f"    - {pid}: delta = {delta_val:.6f}")

# Zapisz wyniki Drift-E2 do plików
DRIFT_E2_OUT_DIR = os.path.join(OUT_ROOT, "experiments", "Drift_E2")
ensure_dir(DRIFT_E2_OUT_DIR)
E2_CONF_DRIFT_DF.to_csv(os.path.join(DRIFT_E2_OUT_DIR, "Drift_E2_conformance_drift.csv"), index=False)
save_json(os.path.join(DRIFT_E2_OUT_DIR, "Drift_E2_conformance_drop_nodes.json"), E2_CONFORMANCE_DROP_NODES)
print(f"\n[Drift-E2] ✓ Zapisano wyniki do {DRIFT_E2_OUT_DIR}/")


[Drift-E2] Uruchamianie eksperymentu na 12 konfiguracjach...

[Drift-E2] [1/12] log=running_example noise=0.0
      ... MC progress 50/300 (players=6, cached=63) [0.0s]
      ... MC progress 100/300 (players=6, cached=64) [0.0s]
      ... MC progress 150/300 (players=6, cached=64) [0.0s]
      ... MC progress 200/300 (players=6, cached=64) [0.0s]
      ... MC progress 250/300 (players=6, cached=64) [0.0s]
      ... MC progress 300/300 (players=6, cached=64) [0.0s]
      ... MC progress 50/300 (players=6, cached=64) [0.0s]
      ... MC progress 100/300 (players=6, cached=64) [0.0s]
      ... MC progress 150/300 (players=6, cached=64) [0.0s]
      ... MC progress 200/300 (players=6, cached=64) [0.0s]
      ... MC progress 250/300 (players=6, cached=64) [0.0s]
      ... MC progress 300/300 (players=6, cached=64) [0.0s]
      ... MC progress 50/300 (players=6, cached=62) [0.0s]
      ... MC progress 100/300 (players=6, cached=64) [0.0s]
      ... MC progress 150/300 (players=6, cached=64) 

,log,noise,window,fitness_delta,mean_phi_delta,max_phi_delta,corr_delta,runtime_s_window,top_nodes_by_delta,nodes_responsible_for_drop
0,running_example,0.00,0->1,0.048227,0.005556,0.013333,NaN,0.025047,"[Seq2@4, Seq2@5, Seq2@3, Seq2@2@2, Seq2@2@1, S...",[]
1,running_example,0.00,1->2,-0.027574,0.010000,0.030000,NaN,0.015590,"[Seq2@4, Seq2@5, Seq2@3, Seq2@2@2, Seq2@2@1, S...","[Seq2@5, Seq2@4]"
2,running_example,0.00,2->3,-0.046015,0.007778,0.016667,NaN,0.015457,"[Seq2@2@2, Seq2@4, Seq2@3, Seq2@5, Seq2@2@1, S...","[Seq2@4, Seq2@2@2]"
3,running_example,0.00,3->4,-0.026164,0.025556,0.060000,NaN,0.015098,"[Seq2@2@2, Seq2@5, Seq2@4, Seq2@3, Seq2@2@1, S...","[Seq2@5, Seq2@2@2]"
4,running_example,0.00,4->5,0.095292,0.026667,0.080000,NaN,0.014916,"[Seq2@2@2, Seq2@5, Seq2@4, Seq2@3, Seq2@2@1, S...",[]
5,running_example,0.25,0->1,0.158946,0.021111,0.060000,NaN,0.014996,"[Seq2@5, Seq2@4, Seq2@2@2, Seq2@3, Seq2@2@1, S...",[]
6,running_example,0.25,1->2,0.018665,0.016667,0.040000,NaN,0.017538,"[Seq2@4, Seq2@5, Seq2@2@2, Seq2@3, Seq2@2@1, S...",[]
7,running_example,0.25,2->3,-0.229977,0.012222,0.036667,NaN,0.021115,"[Seq2@2@2, Seq2@3, Seq2@5, Seq2@4, Seq2@2@1, S...","[Seq2@3, Seq2@2@2]"
8,running_example,0.25,3->4,0.228683,0.010000,0.026667,NaN,0.015134,"[Seq2@5, Seq2@3, Seq2@2@2, Seq2@4, Seq2@2@1, S...",[]
9,running_example,0.25,4->5,-0.211124,0.014444,0.026667,0.000000e+00,0.015642,"[Seq2@3, Seq2@2@2, Seq2@5, Seq2@4, Seq2@2@1, S...","[Seq2@3, Seq2@2@2]"



=== SZCZEGÓŁY WĘZŁÓW ODPOWIEDZIALNYCH ZA SPADEK KONFORMACJI (top 5 z największym spadkiem) ===

running_example_0.25_2->3:
  Spadek konformacji (fitness_delta): -0.229977
  Liczba węzłów odpowiedzialnych za spadek: 2
  Top 5 węzłów z najwyższymi deltami:
    - Seq2@2@2: delta = 0.036667
    - Seq2@3: delta = 0.023333
    - Seq2@5: delta = 0.010000
    - Seq2@4: delta = 0.003333
    - Seq2@2@1: delta = 0.000000

bpi_2012_0.5_1->2:
  Spadek konformacji (fitness_delta): -0.229282
  Liczba węzłów odpowiedzialnych za spadek: 8
  Top 5 węzłów z najwyższymi deltami:
    - Seq2@5@1: delta = 0.050000
    - Seq2@7@1: delta = 0.026667
    - Seq2@9@1: delta = 0.026667
    - Seq2@15@1: delta = 0.023333
    - Seq2@6@1: delta = 0.020000

hospital_billing_0.25_3->4:
  Spadek konformacji (fitness_delta): -0.216648
  Liczba węzłów odpowiedzialnych za spadek: 9
  Top 5 węzłów z najwyższymi deltami:
    - Seq2@7: delta = 0.036667
    - Seq2@13@2: delta = 0.030000
    - Seq2@2@1: delta = 0.026667
    - Se

#### Experiment 3 – Shapley-Guided Simplification

In [4]:
from Functions.coalition import build_coalition_artifacts
from Functions.properties import evaluate_property

def _fetch_shapley_values(log_name: str, noise: float, property_type: str):
    if "SHAPLEY_RESULTS" not in globals():
        raise ValueError("Uruchom najpierw Shapley_mining.ipynb")
    for row in SHAPLEY_RESULTS:
        if row["log"] == log_name and row["noise"] == noise and row["property"] == property_type:
            return row["phi_mc"], row["mc_meta"].get("seconds", 0.0)
    raise ValueError(f"No Shapley results for {(log_name, noise, property_type)}")

# E3: Wszystkie konfiguracje - wszystkie logi, wszystkie poziomy szumu, wszystkie właściwości
E3_SIMPLIFY_CONFIGS = [
    {"log": log_name, "noise": noise, "property": prop}
    for log_name in LOG_PATHS.keys()
    for noise in NOISE_LEVELS
    for prop in ["satisfiability", "liveness", "safety"]
]
E3_SIMPLIFY_PARAMS = {
    "thresholds": [0.01, 0.03, 0.05, 0.1],
}

E3_SIMPLIFY_ROWS = []

total_configs = len(E3_SIMPLIFY_CONFIGS)
print(f"[Drift-E3] Uruchamianie eksperymentu na {total_configs} konfiguracjach...")

for idx, cfg in enumerate(E3_SIMPLIFY_CONFIGS, 1):
    log_name = cfg["log"]
    noise = cfg["noise"]
    prop = cfg["property"]
    
    try:
        expr = PATTERNS[(log_name, noise)]
        base_phi, _ = _fetch_shapley_values(log_name, noise, prop)

        print(f"\n[Drift-E3] [{idx}/{total_configs}] log={log_name} noise={noise} property={prop}")
        for theta in E3_SIMPLIFY_PARAMS["thresholds"]:
            keep_ids = {pid for pid, val in base_phi.items() if abs(val) >= theta}
            masked_expr, spec_text, ini, fin = build_coalition_artifacts(expr, keep_ids, TEMPLATES)
            sat_ok = bool(evaluate_property(spec_text, "satisfiability"))
            liv_ok = bool(evaluate_property(spec_text, "liveness", ini, fin))
            saf_ok = bool(evaluate_property(spec_text, "safety", ini, fin))
            drift = sum(abs(base_phi.get(pid, 0.0) - (base_phi.get(pid, 0.0) if pid in keep_ids else 0.0)) for pid in base_phi) / len(base_phi) if base_phi else 0.0
            E3_SIMPLIFY_ROWS.append({
                "log": log_name,
                "noise": noise,
                "property": prop,
                "threshold": theta,
                "kept_nodes": len(keep_ids),
                "sat_ok": sat_ok,
                "liv_ok": liv_ok,
                "saf_ok": saf_ok,
                "mean_phi_shift": drift
            })
        
        print(f"[Drift-E3] [{idx}/{total_configs}] ✓ zakończono")
    except Exception as e:
        print(f"[Drift-E3] [{idx}/{total_configs}] ✗ błąd: {e}")
        continue

E3_SIMPLIFY_DF = pd.DataFrame(E3_SIMPLIFY_ROWS)
print(f"\n[Drift-E3] Zakończono eksperyment. Wyniki dla {len(E3_SIMPLIFY_ROWS)} kombinacji.")
display(E3_SIMPLIFY_DF)

# Zapisz wyniki Drift-E3 do plików
DRIFT_E3_OUT_DIR = os.path.join(OUT_ROOT, "experiments", "Drift_E3")
ensure_dir(DRIFT_E3_OUT_DIR)
E3_SIMPLIFY_DF.to_csv(os.path.join(DRIFT_E3_OUT_DIR, "Drift_E3_simplification.csv"), index=False)
print(f"\n[Drift-E3] ✓ Zapisano wyniki do {DRIFT_E3_OUT_DIR}/")


[Drift-E3] Uruchamianie eksperymentu na 36 konfiguracjach...

[Drift-E3] [1/36] log=running_example noise=0.0 property=satisfiability
[Drift-E3] [1/36] ✓ zakończono

[Drift-E3] [2/36] log=running_example noise=0.0 property=liveness
[Drift-E3] [2/36] ✓ zakończono

[Drift-E3] [3/36] log=running_example noise=0.0 property=safety
[Drift-E3] [3/36] ✓ zakończono

[Drift-E3] [4/36] log=running_example noise=0.25 property=satisfiability
[Drift-E3] [4/36] ✓ zakończono

[Drift-E3] [5/36] log=running_example noise=0.25 property=liveness
[Drift-E3] [5/36] ✓ zakończono

[Drift-E3] [6/36] log=running_example noise=0.25 property=safety
[Drift-E3] [6/36] ✓ zakończono

[Drift-E3] [7/36] log=running_example noise=0.5 property=satisfiability
[Drift-E3] [7/36] ✓ zakończono

[Drift-E3] [8/36] log=running_example noise=0.5 property=liveness
[Drift-E3] [8/36] ✓ zakończono

[Drift-E3] [9/36] log=running_example noise=0.5 property=safety
[Drift-E3] [9/36] ✓ zakończono

[Drift-E3] [10/36] log=running_example no

,log,noise,property,threshold,kept_nodes,sat_ok,liv_ok,saf_ok,mean_phi_shift
0,running_example,0.0,satisfiability,0.01,4,True,True,False,0.000000
1,running_example,0.0,satisfiability,0.03,4,True,True,False,0.000000
2,running_example,0.0,satisfiability,0.05,4,True,True,False,0.000000
3,running_example,0.0,satisfiability,0.10,3,True,True,False,0.012333
4,running_example,0.0,liveness,0.01,0,True,True,False,0.000000
...,...,...,...,...,...,...,...,...,...
139,bpi_2012,1.0,liveness,0.10,0,True,True,False,0.000000
140,bpi_2012,1.0,safety,0.01,0,True,True,False,0.000000
141,bpi_2012,1.0,safety,0.03,0,True,True,False,0.000000
142,bpi_2012,1.0,safety,0.05,0,True,True,False,0.000000



[Drift-E3] ✓ Zapisano wyniki do ../Docs/Problems/shapley_values/experiments/Drift_E3/


#### Experiment 4 – Robustness to Noise


In [ ]:
# E4: Wszystkie logi, wszystkie właściwości, wszystkie poziomy szumu (baseline: noise=0.0)
E4_CONFIGS = [
    {"log": log_name, "property": prop}
    for log_name in LOG_PATHS.keys()
    for prop in ["satisfiability", "liveness", "safety"]
]
E4_NOISE_LEVELS = NOISE_LEVELS

E4_NOISE_DRIFT_ROWS = []
E4_NOISE_NODES_DETAILS = {}

total_configs = len(E4_CONFIGS)
print(f"[Drift-E4] Uruchamianie eksperymentu na {total_configs} konfiguracjach...")

for idx, cfg in enumerate(E4_CONFIGS, 1):
    log_name = cfg["log"]
    prop = cfg["property"]
    
    try:
        print(f"\n[Drift-E4] [{idx}/{total_configs}] log={log_name} property={prop}")
        print(f"  Baseline: noise=0.0")

        try:
            base_phi_e4, _ = _fetch_shapley_values(log_name, 0.0, prop)
            print("  ✓ Załadowano baseline z cache")
        except ValueError:
            expr_base = PATTERNS[(log_name, 0.0)]
            base_phi_e4, _ = shapley_mc_permutations(
                expr_base,
                TEMPLATES,
                prop,
                n_perm=300,
                seed=4000 + idx * 100,
                progress_every=50,
            )
            print("  ✓ Obliczono baseline")

        for noise in E4_NOISE_LEVELS:
            matched_noise = noise if noise in NOISE_LEVELS else min(NOISE_LEVELS, key=lambda x: abs(x - noise))
            expr_noise = PATTERNS[(log_name, matched_noise)]
            
            try:
                phi_noise, _ = _fetch_shapley_values(log_name, matched_noise, prop)
            except ValueError:
                phi_noise, _ = shapley_mc_permutations(
                    expr_noise,
                    TEMPLATES,
                    prop,
                    n_perm=300,
                    seed=int(noise * 10000) + idx * 100 + 123,
                    progress_every=50,
                )p
            
            deltas_dict = {pid: abs(phi_noise.get(pid, 0.0) - base_phi_e4.get(pid, 0.0)) for pid in base_phi_e4.keys()}
            delta_avg = sum(deltas_dict.values()) / len(deltas_dict) if deltas_dict else 0.0
            
            sorted_nodes_by_delta = sorted(deltas_dict.items(), key=lambda x: x[1], reverse=True)
            top_changed_nodes = sorted_nodes_by_delta[:10]
            
            config_noise_key = f"{log_name}_{prop}_{noise}"
            E4_NOISE_DRIFT_ROWS.append({
                "log": log_name,
                "property": prop,
                "noise_added": noise,
                "matched_noise": matched_noise,
                "delta_vs_base": delta_avg,
                "max_delta": max(deltas_dict.values()) if deltas_dict else 0.0,
                "top_changed_nodes": [pid for pid, _ in top_changed_nodes],
            })
            
            E4_NOISE_NODES_DETAILS[config_noise_key] = {
                "log": log_name,
                "property": prop,
                "noise": noise,
                "delta_avg": delta_avg,
                "top_changed_nodes": top_changed_nodes,
                "all_deltas": deltas_dict
            }
        
        print(f"[Drift-E4] [{idx}/{total_configs}] ✓ zakończono")
    except Exception as e:
        print(f"[Drift-E4] [{idx}/{total_configs}] ✗ błąd: {e}")
        continue

E4_NOISE_DF = pd.DataFrame(E4_NOISE_DRIFT_ROWS)
print(f"\n[Drift-E4] Zakończono eksperyment. Wyniki dla {len(E4_NOISE_DRIFT_ROWS)} kombinacji.")
display(E4_NOISE_DF)

print("\n=== SZCZEGÓŁY ZMIAN WĘZŁÓW PRZY RÓŻNYCH POZIOMACH SZUMU (top 5 z największymi zmianami) ===")
if E4_NOISE_DRIFT_ROWS:
    top_changes = E4_NOISE_DF.nlargest(5, "delta_vs_base")
    for _, row in top_changes.iterrows():
        if row["noise_added"] == 0.0:
            continue
        config_key = f"{row['log']}_{row['property']}_{row['noise_added']}"
        if config_key in E4_NOISE_NODES_DETAILS:
            details = E4_NOISE_NODES_DETAILS[config_key]
            print(f"\n{config_key}:")
            print(f"  Średnia delta względem baseline: {details['delta_avg']:.6f}")
            print(f"  Top 5 węzłów z największymi zmianami:")
            for pid, delta_val in details["top_changed_nodes"][:5]:
                print(f"    - {pid}: delta = {delta_val:.6f}")

# Zapisz wyniki Drift-E4 do plików
DRIFT_E4_OUT_DIR = os.path.join(OUT_ROOT, "experiments", "Drift_E4")
ensure_dir(DRIFT_E4_OUT_DIR)
E4_NOISE_DF.to_csv(os.path.join(DRIFT_E4_OUT_DIR, "Drift_E4_noise_robustness.csv"), index=False)
save_json(os.path.join(DRIFT_E4_OUT_DIR, "Drift_E4_noise_nodes_details.json"), E4_NOISE_NODES_DETAILS)
print(f"\n[Drift-E4] ✓ Zapisano wyniki do {DRIFT_E4_OUT_DIR}/")


[Drift-E4] Uruchamianie eksperymentu na 9 konfiguracjach...

[Drift-E4] [1/9] log=running_example property=satisfiability
  Baseline: noise=0.0
  ✓ Załadowano baseline z cache
[Drift-E4] [1/9] ✓ zakończono

[Drift-E4] [2/9] log=running_example property=liveness
  Baseline: noise=0.0
  ✓ Załadowano baseline z cache
[Drift-E4] [2/9] ✓ zakończono

[Drift-E4] [3/9] log=running_example property=safety
  Baseline: noise=0.0
  ✓ Załadowano baseline z cache
[Drift-E4] [3/9] ✓ zakończono

[Drift-E4] [4/9] log=hospital_billing property=satisfiability
  Baseline: noise=0.0
  ✓ Załadowano baseline z cache
[Drift-E4] [4/9] ✓ zakończono

[Drift-E4] [5/9] log=hospital_billing property=liveness
  Baseline: noise=0.0
  ✓ Załadowano baseline z cache
[Drift-E4] [5/9] ✓ zakończono

[Drift-E4] [6/9] log=hospital_billing property=safety
  Baseline: noise=0.0
  ✓ Załadowano baseline z cache
[Drift-E4] [6/9] ✓ zakończono

[Drift-E4] [7/9] log=bpi_2012 property=satisfiability
  Baseline: noise=0.0
  ✓ Załadowa

,log,property,noise_added,matched_noise,delta_vs_base,max_delta,top_changed_nodes
0,running_example,satisfiability,0.00,0.00,0.000000,0.000,"[Seq2@5, Seq2@4, Seq2@3, Seq2@2@1, Seq2@2@2, S..."
1,running_example,satisfiability,0.25,0.25,0.000000,0.000,"[Seq2@5, Seq2@4, Seq2@3, Seq2@2@1, Seq2@2@2, S..."
2,running_example,satisfiability,0.50,0.50,0.000000,0.000,"[Seq2@5, Seq2@4, Seq2@3, Seq2@2@1, Seq2@2@2, S..."
3,running_example,satisfiability,1.00,1.00,0.000000,0.000,"[Seq2@5, Seq2@4, Seq2@3, Seq2@2@1, Seq2@2@2, S..."
4,running_example,liveness,0.00,0.00,0.000000,0.000,"[Seq2@5, Seq2@4, Seq2@3, Seq2@2@1, Seq2@2@2, S..."
5,running_example,liveness,0.25,0.25,0.000000,0.000,"[Seq2@5, Seq2@4, Seq2@3, Seq2@2@1, Seq2@2@2, S..."
6,running_example,liveness,0.50,0.50,0.000000,0.000,"[Seq2@5, Seq2@4, Seq2@3, Seq2@2@1, Seq2@2@2, S..."
7,running_example,liveness,1.00,1.00,0.000000,0.000,"[Seq2@5, Seq2@4, Seq2@3, Seq2@2@1, Seq2@2@2, S..."
8,running_example,safety,0.00,0.00,0.000000,0.000,"[Seq2@5, Seq2@4, Seq2@3, Seq2@2@1, Seq2@2@2, S..."
9,running_example,safety,0.25,0.25,0.000000,0.000,"[Seq2@5, Seq2@4, Seq2@3, Seq2@2@1, Seq2@2@2, S..."



=== SZCZEGÓŁY ZMIAN WĘZŁÓW PRZY RÓŻNYCH POZIOMACH SZUMU (top 5 z największymi zmianami) ===

bpi_2012_liveness_0.5:
  Średnia delta względem baseline: 0.033933
  Top 5 węzłów z największymi zmianami:
    - Seq2@22@3: delta = 0.124000
    - Seq2@7@1: delta = 0.122000
    - Seq2@6@1: delta = 0.107000
    - Seq2@16@3: delta = 0.105000
    - Seq2@15@2: delta = 0.101000

hospital_billing_liveness_0.5:
  Średnia delta względem baseline: 0.032432
  Top 5 węzłów z największymi zmianami:
    - Seq2@6@2: delta = 0.322000
    - Seq2@2@1: delta = 0.279000
    - Seq2@15@2: delta = 0.096000
    - Seq2@21@4: delta = 0.092000
    - Seq2@19@2: delta = 0.049000

bpi_2012_satisfiability_0.5:
  Średnia delta względem baseline: 0.032000
  Top 5 węzłów z największymi zmianami:
    - Seq2@22@3: delta = 0.129000
    - Seq2@7@1: delta = 0.113000
    - Seq2@6@1: delta = 0.103000
    - Seq2@16@3: delta = 0.095000
    - Seq2@15@2: delta = 0.093000

bpi_2012_liveness_0.25:
  Średnia delta względem baseline: 0.031